# 02 - Model 1: Denial Classification

We want to guess, before a claim is sent, if it will be denied.

* Target: is_denied (1 = denied, 0 = paid or partial_pay).
* pending claims are left out because we do not know their result yet.
* Model used: a tuned XGBoost classifier inside a Scikit-learn pipeline.
* We only use the 5 features picked in notebook 01.
* Tuning: RandomizedSearchCV (n_iter=10) with 3-fold stratified CV.

Saved in models/: denial_model.pkl and results added to models/model_metadata.json.


## 1. Imports and paths


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve, classification_report)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

RANDOM_STATE = 42
BASE = Path.cwd()
while not (BASE / "data" / "processed").exists() and BASE != BASE.parent:
    BASE = BASE.parent
PROCESSED = BASE / "data" / "processed"
MODELS = BASE / "models"
REPORTS = BASE / "reports"
MODELS.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)
print("Project root:", BASE)


Project root: d:\CDAC\Projects\ML_Project_Final\medical-claim-denial-predictor


## 2. Load the model-ready dataset


In [2]:
df = pd.read_csv(PROCESSED / "claims_model_ready.csv")
with open(PROCESSED / "feature_columns.json", encoding="utf-8") as fh:
    cols = json.load(fh)
# only use the features picked in notebook 01
CATEGORICAL = cols["selected_categorical"]
NUMERIC = cols["selected_numeric"]
FEATURES = cols["selected_features"]
print("Selected features (%d):" % len(FEATURES), FEATURES)
print("Rows loaded:", df.shape)


Selected features (5): ['documentation_completeness', 'cpt_code', 'primary_icd10_dx', 'claim_amount_usd', 'auth_status']
Rows loaded: (120000, 40)


## 3. Pick features and target, split train / val / test


In [3]:
data = df[df["is_denied"].notna()].copy()          # drop the pending rows
data["is_denied"] = data["is_denied"].astype(int)
X = data[FEATURES]
y = data["is_denied"]

train_rows = data["split"] == "train"
val_rows = data["split"] == "val"
test_rows = data["split"] == "test"

X_train = X[train_rows]
y_train = y[train_rows]
X_val = X[val_rows]
y_val = y[val_rows]
X_test = X[test_rows]
y_test = y[test_rows]
print("train/val/test:", X_train.shape, X_val.shape, X_test.shape)
print("denial rate  :", round(y_train.mean(), 4))


train/val/test: (78081, 5) (16722, 5) (16709, 5)
denial rate  : 0.3022


## 4. Preprocessing step

* Text columns -> OneHotEncoder (it ignores values it has never seen).
* Number columns -> fill missing values with the median, then scale.

This all happens inside the pipeline, so it is fitted on the train data only and
saved together with the model.


In [4]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC),
    ("cat", categorical_pipe, CATEGORICAL),
])


## 5. Try many settings (RandomizedSearchCV, n_iter = 10)

We use scale_pos_weight because there are more paid claims than denied ones.


In [5]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
base_spw = neg / pos
print("scale_pos_weight base:", round(base_spw, 3))

pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )),
])

param_dist = {
    "clf__n_estimators": [200, 300, 400, 600],
    "clf__max_depth": [3, 4, 5, 6, 8],
    "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "clf__subsample": [0.7, 0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "clf__min_child_weight": [1, 3, 5],
    "clf__gamma": [0, 0.1, 0.3],
    "clf__reg_alpha": [0, 0.1, 1],
    "clf__reg_lambda": [1, 1.5, 2],
    "clf__scale_pos_weight": [1, round(base_spw, 2), round(base_spw * 1.5, 2)],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
search.fit(X_train, y_train)
print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Best params    :", search.best_params_)


scale_pos_weight base: 2.309


Best CV ROC-AUC: 0.9854
Best params    : {'clf__subsample': 0.9, 'clf__scale_pos_weight': np.float64(3.46), 'clf__reg_lambda': 1.5, 'clf__reg_alpha': 0, 'clf__n_estimators': 200, 'clf__min_child_weight': 3, 'clf__max_depth': 5, 'clf__learning_rate': 0.03, 'clf__gamma': 0.3, 'clf__colsample_bytree': 0.6}


## 6. Check on the validation set, then test on the test set


In [6]:
best = search.best_estimator_
val_proba = best.predict_proba(X_val)[:, 1]
print("Validation ROC-AUC:", round(roc_auc_score(y_val, val_proba), 4))
print("Validation PR-AUC :", round(average_precision_score(y_val, val_proba), 4))


Validation ROC-AUC: 0.9848
Validation PR-AUC : 0.9666


In [7]:
proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

metrics = {
    "accuracy": round(accuracy_score(y_test, pred), 4),
    "precision": round(precision_score(y_test, pred), 4),
    "recall": round(recall_score(y_test, pred), 4),
    "f1": round(f1_score(y_test, pred), 4),
    "roc_auc": round(roc_auc_score(y_test, proba), 4),
    "pr_auc": round(average_precision_score(y_test, proba), 4),
    "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    "threshold": 0.5,
    "test_rows": int(len(y_test)),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, pred, target_names=["approved", "denied"]))


{
  "accuracy": 0.9168,
  "precision": 0.7859,
  "recall": 1.0,
  "f1": 0.8801,
  "roc_auc": 0.9857,
  "pr_auc": 0.9694,
  "confusion_matrix": [
    [
      10217,
      1390
    ],
    [
      0,
      5102
    ]
  ],
  "threshold": 0.5,
  "test_rows": 16709
}
              precision    recall  f1-score   support

    approved       1.00      0.88      0.94     11607
      denied       0.79      1.00      0.88      5102

    accuracy                           0.92     16709
   macro avg       0.89      0.94      0.91     16709
weighted avg       0.93      0.92      0.92     16709



## 7. Look at the results with plots


In [8]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["approved", "denied"], yticklabels=["approved", "denied"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Denial confusion matrix")
plt.tight_layout()
plt.savefig(REPORTS / "02_confusion_matrix.png", dpi=110)
plt.show()


C:\Windows\Temp\ipykernel_34828\2426527681.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
fpr, tpr, _ = roc_curve(y_test, proba)
prec, rec, _ = precision_recall_curve(y_test, proba)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr, tpr, label="ROC-AUC = %.3f" % metrics["roc_auc"])
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve")
axes[0].legend()
axes[1].plot(rec, prec, label="PR-AUC = %.3f" % metrics["pr_auc"])
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve")
axes[1].legend()
plt.tight_layout()
plt.savefig(REPORTS / "02_roc_pr.png", dpi=110)
plt.show()


C:\Windows\Temp\ipykernel_34828\1263760729.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Which features matter most


In [10]:
feat_names = best.named_steps["prep"].get_feature_names_out()
importances = best.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"feature": feat_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)
imp_df = imp_df.reset_index(drop=True)
print(imp_df.head(20))

plt.figure(figsize=(8, 6))
sns.barplot(data=imp_df.head(20), y="feature", x="importance")
plt.title("XGBoost feature importance (top 20)")
plt.tight_layout()
plt.savefig(REPORTS / "02_feature_importance.png", dpi=110)
plt.show()


                            feature  importance
0   num__documentation_completeness    0.854291
1     cat__auth_status_not_obtained    0.049607
2     cat__auth_status_not_required    0.020801
3         cat__auth_status_obtained    0.014256
4               cat__cpt_code_27447    0.002689
5               cat__cpt_code_74177    0.002640
6               cat__cpt_code_93454    0.002265
7               cat__cpt_code_29827    0.001222
8               cat__cpt_code_99291    0.001221
9               cat__cpt_code_38220    0.001221
10              cat__cpt_code_99215    0.001181
11              cat__cpt_code_95886    0.001165
12     cat__primary_icd10_dx_J45.41    0.001130
13              cat__cpt_code_99233    0.001125
14      cat__primary_icd10_dx_C18.9    0.001114
15     cat__primary_icd10_dx_Z34.00    0.001075
16      cat__primary_icd10_dx_F20.9    0.001038
17              cat__cpt_code_70553    0.001016
18              cat__cpt_code_78452    0.000996
19              cat__cpt_code_31623    0

C:\Windows\Temp\ipykernel_34828\3857110145.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# permutation importance on a small sample to keep it fast
sample = X_test.sample(min(5000, len(X_test)), random_state=RANDOM_STATE)
perm = permutation_importance(best, sample, y_test.loc[sample.index],
                              n_repeats=5, random_state=RANDOM_STATE,
                              scoring="roc_auc", n_jobs=-1)
perm_df = pd.DataFrame({"feature": FEATURES, "importance": perm.importances_mean})
perm_df = perm_df.sort_values("importance", ascending=False)
perm_df = perm_df.reset_index(drop=True)
print(perm_df.head(15))
plt.figure(figsize=(8, 6))
sns.barplot(data=perm_df.head(20), y="feature", x="importance")
plt.title("Permutation importance (top 20)")
plt.tight_layout()
plt.savefig(REPORTS / "02_permutation_importance.png", dpi=110)
plt.show()


                      feature  importance
0  documentation_completeness    0.437640
1                 auth_status    0.000344
2                    cpt_code    0.000051
3            claim_amount_usd    0.000011
4            primary_icd10_dx   -0.000073


C:\Windows\Temp\ipykernel_34828\1486439419.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Risk bands (used by the API and UI)

The predicted probabilities are mostly very low or very high, so we use three
simple bands instead of percentiles.


In [12]:
risk_bands = {"low_max": 0.33, "medium_max": 0.66}
print("Risk bands:", risk_bands)

# check how the validation set splits into the bands
band_codes = []
for p in val_proba:
    code = np.digitize(p, [risk_bands["low_max"], risk_bands["medium_max"]])
    band_codes.append(code)
val_bands = pd.Series(band_codes)
val_bands = val_bands.value_counts()
val_bands = val_bands.sort_index()
print("validation distribution (0=Low,1=Medium,2=High):", val_bands.to_dict())


def risk_level(p):
    if p <= risk_bands["low_max"]:
        return "Low"
    if p <= risk_bands["medium_max"]:
        return "Medium"
    return "High"


example_levels = []
for p in [0.1, 0.5, 0.9]:
    example_levels.append(risk_level(p))
print("example:", example_levels)


Risk bands: {'low_max': 0.33, 'medium_max': 0.66}


validation distribution (0=Low,1=Medium,2=High): {0: 10344, 1: 106, 2: 6272}
example: ['Low', 'Medium', 'High']


## 10. Save the model and metadata


In [13]:
joblib.dump(best, MODELS / "denial_model.pkl")

top_features = imp_df.head(15)["feature"].tolist()
meta_path = MODELS / "model_metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
else:
    meta = {}

# turn numpy numbers into plain numbers so json can save them
best_params = {}
for k, v in search.best_params_.items():
    if isinstance(v, (np.integer, np.floating)):
        best_params[k] = v.item()
    else:
        best_params[k] = v

meta["denial_classification"] = {
    "algorithm": "XGBoost (binary:logistic)",
    "target": "is_denied (pending excluded)",
    "best_params": best_params,
    "cv_best_roc_auc": round(float(search.best_score_), 4),
    "metrics": metrics,
    "risk_bands": risk_bands,
    "features_used": FEATURES,
    "top_features": top_features,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
}
meta_path.write_text(json.dumps(meta, indent=2))
print("Saved", MODELS / "denial_model.pkl")
print("=== MODEL 1 COMPLETE ===")


Saved d:\CDAC\Projects\ML_Project_Final\medical-claim-denial-predictor\models\denial_model.pkl
=== MODEL 1 COMPLETE ===
